In [4]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from typing import Dict, Tuple
from cargar_dataset import cargar_dataset

def realizar_test_ks_datasets(df_ref: pd.DataFrame, df_eval: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    Realiza el test de Kolmogorov-Smirnov de dos muestras para cada columna común.
    
    Argumentos:
        df_ref: DataFrame de referencia (ej. Datos Reales / Entrenamiento).
        df_eval: DataFrame a evaluar (ej. Datos Sintéticos / Producción).
        alpha: Nivel de significancia (por defecto 0.05).
        
    Retorna:
        DataFrame con el estadístico D, p-valor y si se rechaza la hipótesis nula (H0).
        H0: Las dos muestras provienen de la misma distribución.
    """
    columnas = [col for col in df_ref.columns if col in df_eval.columns]
    resultados = []

    for col in columnas:
        # Extraer datos ignorando NaNs para el test
        data_ref = df_ref[col].dropna()
        data_eval = df_eval[col].dropna()
        
        # Ejecutar KS de dos muestras
        d_stat, p_val = ks_2samp(data_ref, data_eval)
        
        resultados.append({
            "feature": col,
            "ks_statistic_D": round(d_stat, 4),
            "p_value": round(p_val, 6),
            "distribucion_diferente": p_val < alpha
        })
        
    return pd.DataFrame(resultados)

# ======================================================================================
# EJEMPLO DE INTEGRACIÓN (Diagnóstico de Compatibilidad Geométrica)
# ======================================================================================

# 1. Cargamos el dataset original usando tu función 'cargar_dataset'
# Suponiendo que cfg ya está definido en tu config_datasets
from config_datasets import config_datasets

nombre = "shuttle" # Ejemplo
cfg = config_datasets[nombre]

try:
    X_df, y, clases = cargar_dataset(
        path=cfg.get("path"),
        clase_minoria=cfg.get("clase_minoria"),
        col_features=cfg.get("col_features"),
        col_target=cfg.get("col_target"),
        sep=cfg.get("sep", ","),
        header=cfg.get("header", None),
        binarizar=False,
        names=cfg.get("esquema")
    )

    # Simulación de uso para Tesis:
    # Dividimos el dataset para ver si Train y Test son estadísticamente similares
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, stratify=y)

    print(f"\n🧪 Realizando Test K-S para: {nombre.upper()}")
    reporte_ks = realizar_test_ks_datasets(X_train, X_test)
    
    # Mostrar solo las variables que fallan (donde la distribución es diferente)
    drift = reporte_ks[reporte_ks["distribucion_diferente"] == True]
    
    if not drift.empty:
        print(f"⚠️ Se detectó divergencia en {len(drift)} características:")
        print(drift[["feature", "ks_statistic_D", "p_value"]])
    else:
        print("✅ No hay divergencia significativa entre las distribuciones (H0 mantenida).")
        print(f"Distancia D promedio: {reporte_ks['ks_statistic_D'].mean():.4f}")

except Exception as e:
    print(f"❌ Error: {e}")


🧪 Realizando Test K-S para: SHUTTLE
✅ No hay divergencia significativa entre las distribuciones (H0 mantenida).
Distancia D promedio: 0.0067
